# Imports

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [2]:
keras.mixed_precision.set_global_policy('mixed_float16')

In [3]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm.notebook as tqdm
import transformers

c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: True


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [5]:
tokenizer: transformers.PreTrainedTokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

In [6]:
assert isinstance(tokenizer.eos_token_id, int)
PAD_TOKEN_ID: int = tokenizer.eos_token_id + 1

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [7]:
df = pd.read_csv("../../data/rnn_seq2seq/rus-eng.tsv", sep="\t", usecols=[1, 3], names=["ru", "en"])

In [8]:
import pickle

In [9]:
TOKENIZE_NOW = False

In [10]:
if TOKENIZE_NOW:
    tokens_ru = [tokenizer.encode(text) for text in df["ru"]]
    with open("../../data/rnn_seq2seq/ru.pkl", "wb") as f:
        pickle.dump(tokens_ru, f, protocol=pickle.HIGHEST_PROTOCOL)

In [11]:
if TOKENIZE_NOW:
    tokens_en = [tokenizer.encode(text) for text in df["en"]]
    with open("../../data/rnn_seq2seq/en.pkl", "wb") as f:
            pickle.dump(tokens_en, f, protocol=pickle.HIGHEST_PROTOCOL)

In [12]:
with open("../../data/rnn_seq2seq/ru.pkl", "rb") as f:
    tokens_ru = pickle.load(f)

with open("../../data/rnn_seq2seq/en.pkl", "rb") as f:
    tokens_en = pickle.load(f)

In [13]:
tokens_ru_len = [len(tokens) for tokens in tokens_ru]
tokens_en_len = [len(tokens) for tokens in tokens_en]

In [14]:
class Seq2SeqDataset(keras.utils.PyDataset):
    def __init__(
        self,
        input_sequences: list[list[int]],
        max_seq_length_input: int,
        output_sequences: list[list[int]],
        max_seq_length_output: int,
        tokenizer: transformers.PreTrainedTokenizer,
        pad_token_id: int,
        batch_size: int = 128,
        shuffle: bool = True,
        rng: np.random.Generator | None = None,
        **kwargs,
    ):
        super().__init__(**kwargs)

        if rng is None:
            rng = np.random.default_rng()

        self.input = input_sequences
        self.max_seq_lenth_input = max_seq_length_input
        self.output = output_sequences
        self.max_seq_lenth_output = max_seq_length_output
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = rng
        self.tokenizer = tokenizer
        self.pad_token_id = pad_token_id
        self.indices = np.arange(len(input_sequences))

        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_indices = self.indices[start:end]

        encoder_input_tokens = [self.input[i] for i in batch_indices]
        decoder_input_tokens = [
            [tokenizer.bos_token_id] + self.output[i] for i in batch_indices
        ]
        decoder_output_tokens = [
            self.output[i] + [tokenizer.eos_token_id] for i in batch_indices
        ]

        encoder_input = keras.utils.pad_sequences(
            encoder_input_tokens,
            maxlen=self.max_seq_lenth_input,
            padding="post",
            value=PAD_TOKEN_ID,
        )
        decoder_input = keras.utils.pad_sequences(
            decoder_input_tokens,
            maxlen=self.max_seq_lenth_output,
            padding="post",
            value=PAD_TOKEN_ID,
        )
        decoder_output = keras.utils.pad_sequences(
            decoder_output_tokens,
            maxlen=self.max_seq_lenth_output,
            padding="post",
            value=PAD_TOKEN_ID,
        )

        mask = (decoder_output != PAD_TOKEN_ID).astype(np.float32)

        return (encoder_input, decoder_input), decoder_output, mask

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [15]:
train_ds = Seq2SeqDataset(tokens_ru, 67, tokens_en, 18, tokenizer, PAD_TOKEN_ID, batch_size=128)

In [38]:
val_translations = [
    ("Привет, как дела?", "Hi, how are you?"),
    ("Я люблю читать книги.", "I love reading books."),
    ("Сегодня хорошая погода.", "The weather is nice today."),
    ("Он не пришёл на встречу.", "He didn't come to the meeting."),
    ("Завтра мы поедем в Москву.", "We will go to Moscow tomorrow."),
    ("Мы поедем в Москву завтра.", "We will go to Moscow tomorrow."),
    ("Она готовит ужин на кухне.", "She is cooking dinner in the kitchen."),
    ("Мне нравится этот фильм.", "I like this movie."),
    ("Дети играют во дворе.", "The children are playing in the yard."),
    ("Я не знаю, что делать.", "I don't know what to do."),
    ("Этот дом был построен в прошлом году.", "This house was built last year."),
    ("В прошлом году был построен этот дом.", "This house was built last year."),
    ("Ты знаешь как подстричься?", "Do you know how to get a haircut?"),
    ("Ты знаешь как пройти на пляж?","Do you know how to get to a beach?"),
    ("Ты знаешь как ездить на лошади?", "Do you know how to ride a horse?")
]

In [17]:
val_ru_tokens = []
val_en_tokens = []

for ru, en in val_translations:
    val_ru_tokens.append(tokenizer.encode(ru))
    val_en_tokens.append(tokenizer.encode(en))

In [18]:
val_ds = Seq2SeqDataset(val_ru_tokens, 67, val_en_tokens, 18, tokenizer, PAD_TOKEN_ID, batch_size=10)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [19]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    rnn_type: type[keras.layers.Layer] = keras.layers.LSTM,
) -> keras.Model:
    enc_inputs = keras.Input(shape=(None,), dtype="int32", name="encoder_inputs")
    dec_inputs = keras.Input(shape=(None,), dtype="int32", name="decoder_inputs")

    emb = keras.layers.Embedding(n_tokens, units, name="embedding")

    x = emb(enc_inputs)
    enc_outputs = None
    enc_states = None

    for i in range(n_stacks):
        is_last = i == n_stacks - 1
        layer_name = f"encoder_{i}"

        if bidirectional and is_last:
            bi = keras.layers.Bidirectional(
                rnn_type(units=units, return_sequences=True, return_state=True),
                name=layer_name,
            )
            enc_outputs, *states = bi(x)

            enc_outputs = keras.layers.Dense(units, name="enc_output_proj")(enc_outputs)

            n_dir = len(states) // 2
            enc_states = [
                keras.layers.Dense(
                    units,
                    activation="tanh",
                    name=f"enc_state_proj_{i}_{j}",
                )(keras.layers.Concatenate()([states[j], states[n_dir + j]]))
                for j in range(n_dir)
            ]
        else:
            rnn = rnn_type(
                units=units, return_sequences=True, return_state=True, name=layer_name
            )
            enc_outputs, *enc_states = rnn(x)

    y = emb(dec_inputs)
    dec_outputs = None

    for i in range(n_stacks):
        rnn = rnn_type(
            units=units, return_sequences=True, return_state=True, name=f"decoder_{i}"
        )
        if i == 0:
            dec_outputs, *_ = rnn(y, initial_state=enc_states)
        else:
            dec_outputs, *_ = rnn(y)

    context = keras.layers.AdditiveAttention(name="attention")(
        [dec_outputs, enc_outputs]
    )

    combined = keras.layers.Concatenate(name="concat_context")([dec_outputs, context])

    logits = keras.layers.Dense(n_labels, dtype="float32", name="output")(combined)

    return keras.Model([enc_inputs, dec_inputs], logits, name=name)

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

In [20]:
model = get_model(
    units=256,
    n_tokens=tokenizer.vocab_size + 1,
    n_labels=tokenizer.vocab_size + 1,
    n_stacks=1,
    bidirectional=False,
    name='seq2seq_lstm',
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=['accuracy'],
)

In [21]:
model.summary()

Model: "seq2seq_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │ 12,866,048 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │ decoder_inputs[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0 (LSTM)    │ [(None, None,     │    525,312 │ embedding[0][0]   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_0 (LSTM)    │ [(None, None,     │    525,312 │ embedding[1][0],  │
│                     │ 256), (None,      │            │ encoder_0[0][1],  │
│                     │ 256), (None,      │            │ encoder_0[0][2]   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 256) │        256 │ decoder_0[0][0],  │
│ (AdditiveAttention) │                   │            │ encoder_0[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_context      │ (None, None, 512) │          0 │ decoder_0[0][0],  │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, None,      │ 25,782,354 │ concat_context[0… │
│                     │ 50258)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 39,699,282 (151.44 MB)

 Trainable params: 39,699,282 (151.44 MB)

 Non-trainable params: 0 (0.00 B)

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

In [22]:
from pathlib import Path

In [23]:
LOGS_DIR = Path('../../logs/rnn_seq2seq')
CHECKPOINT_DIR = Path('../../model_checkpoints/rnn_seq2seq')

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [24]:
from datetime import datetime

In [25]:
from torch.utils.tensorboard import SummaryWriter


class PyTorchTensorBoard(keras.callbacks.Callback):
    def __init__(self, log_dir):
        super().__init__()
        self.writer = SummaryWriter(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        if logs:
            for key, value in logs.items():
                self.writer.add_scalar(key, value, epoch)

In [26]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_DIR / f"model_train_accuracy_{datetime.now().strftime('%Y%m%d-%H%M%S')}.keras",
        monitor="accuracy",
        mode="max",
        save_best_only=True,
    ),
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_DIR / f"model_val_accuracy_{datetime.now().strftime('%Y%m%d-%H%M%S')}.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
    ),  
    PyTorchTensorBoard(LOGS_DIR),
]

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

Epoch 1/5


c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:651: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, (h_n, c_n) = torch._VF.lstm(


6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1764s 272ms/step - accuracy: 0.5761 - loss: 1.1738 - val_accuracy: 0.8846 - val_loss: 0.2045
Epoch 2/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1847s 285ms/step - accuracy: 0.7607 - loss: 0.5487 - val_accuracy: 0.9103 - val_loss: 0.1386
Epoch 3/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 2153s 332ms/step - accuracy: 0.7964 - loss: 0.4352 - val_accuracy: 0.8974 - val_loss: 0.1186
Epoch 4/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 2196s 339ms/step - accuracy: 0.8163 - loss: 0.3756 - val_accuracy: 0.8974 - val_loss: 0.1087
Epoch 5/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 2228s 343ms/step - accuracy: 0.8305 - loss: 0.3365 - val_accuracy: 0.9231 - val_loss: 0.1121


In [ ]:
model: keras.models.Model = keras.models.load_model(CHECKPOINT_DIR / "model_val_accuracy_20261017-180838.keras/") # pyright: ignore[reportAssignmentType]

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [ ]:
embed   = model.get_layer("embedding")
enc_lstm  = model.get_layer("encoder_0")
dec_lstm  = model.get_layer("decoder_0")
attn      = model.get_layer("attention")
out_dense = model.get_layer("output")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BOS = tokenizer.bos_token_id
EOS = tokenizer.eos_token_id
PAD = PAD_TOKEN_ID

In [ ]:
@torch.no_grad()
def encode_step(text):
    ids = tokenizer(text, add_special_tokens=False, return_tensors="np")["input_ids"]
    x = torch.as_tensor(ids, dtype=torch.int32, device=device)

    y = embed(x)
    enc_out, h, c = enc_lstm(y, training=False)
    return enc_out, h, c


@torch.no_grad()
def decode_step(last_id, enc_out, h, c):
    last = torch.tensor([[last_id]], dtype=torch.int32, device=device)

    y = embed(last)
    y, h, c = dec_lstm(y, initial_state=[h, c], training=False)

    ctx = attn([y, enc_out], training=False)
    combined = keras.ops.concatenate([y, ctx], axis=-1)
    logits = out_dense(combined, training=False)
    return logits, h, c

In [36]:
def translate(text, max_len=50):
    enc_out, h, c = encode_step(text)

    tokens = []
    next_id = BOS
    for _ in range(max_len):
        logits, h, c = decode_step(next_id, enc_out, h, c)
        next_id = int(keras.ops.convert_to_numpy(keras.ops.argmax(logits[0, -1])))
        if next_id in (EOS, PAD):
            break
        tokens.append(next_id)

    return tokenizer.decode(tokens, skip_special_tokens=True)

In [39]:
for ru, en in val_translations:
    pred = translate(ru)
    print(f"RU  : {ru}")
    print(f"REF : {en}")
    print(f"PRED: {pred}")
    print("-" * 60)

RU  : Привет, как дела?
REF : Hi, how are you?
PRED: Hi, how are you doing?
------------------------------------------------------------
RU  : Я люблю читать книги.
REF : I love reading books.
PRED: I like reading books.
------------------------------------------------------------
RU  : Сегодня хорошая погода.
REF : The weather is nice today.
PRED: Today today is a good weather.
------------------------------------------------------------
RU  : Он не пришёл на встречу.
REF : He didn't come to the meeting.
PRED: He didn't come to the meeting.
------------------------------------------------------------
RU  : Завтра мы поедем в Москву.
REF : We will go to Moscow tomorrow.
PRED: Tomorrow we'll go to Moscow tomorrow.
------------------------------------------------------------
RU  : Мы поедем в Москву завтра.
REF : We will go to Moscow tomorrow.
PRED: We'll go to Moscow tomorrow morning.
------------------------------------------------------------
RU  : Она готовит ужин на кухне.
REF : She

In [41]:
def beam_search_translate(text, beam_width=5, max_len=50, length_penalty=0.7):
    enc_out, h, c = encode_step(text)

    def length_score(tokens, lp):
        n = len(tokens)
        return lp / (((5 + n) / 6) ** length_penalty)

    beams = [
        {
            "tokens": [BOS],
            "h": h,
            "c": c,
            "lp": 0.0,
            "done": False,
        }
    ]
    finished = []

    for _ in range(max_len):
        active = [b for b in beams if not b["done"]]
        if not active:
            break

        K = len(active)

        last_ids = torch.tensor(
            [[b["tokens"][-1]] for b in active],
            dtype=torch.int32,
            device=device,
        )
        h_b = torch.cat([b["h"] for b in active], dim=0)
        c_b = torch.cat([b["c"] for b in active], dim=0)

        with torch.no_grad():
            y = dec_emb(last_ids)
            y, h_new, c_new = dec_lstm(y, initial_state=[h_b, c_b], training=False)
            enc_out_b = enc_out.expand(K, -1, -1)
            ctx = attn([y, enc_out_b], training=False)
            combined = keras.ops.concatenate([y, ctx], axis=-1)
            logits = out_dense(combined, training=False)

        log_probs = keras.ops.convert_to_numpy(
            keras.ops.log_softmax(logits[:, -1, :], axis=-1)
        )

        new_beams = []
        for k, beam in enumerate(active):
            topk = np.argsort(log_probs[k])[-beam_width:]
            for tok in topk:
                tok = int(tok)
                lp = beam["lp"] + float(log_probs[k, tok])
                new = {
                    "tokens": beam["tokens"] + [tok],
                    "h": h_new[k : k + 1],
                    "c": c_new[k : k + 1],
                    "lp": lp,
                    "done": (tok == EOS),
                }
                if new["done"]:
                    finished.append(new)
                else:
                    new_beams.append(new)

        new_beams.sort(key=lambda b: length_score(b["tokens"], b["lp"]), reverse=True)
        beams = new_beams[:beam_width]

    pool = finished if finished else beams
    if not pool:
        return ""
    pool.sort(key=lambda b: length_score(b["tokens"], b["lp"]), reverse=True)

    tokens = pool[0]["tokens"][1:]
    if tokens and tokens[-1] == EOS:
        tokens = tokens[:-1]
    return tokenizer.decode(tokens, skip_special_tokens=True)

In [42]:
for ru, en in val_translations:
    pred = beam_search_translate(ru)
    print(f"RU  : {ru}")
    print(f"REF : {en}")
    print(f"PRED: {pred}")
    print("-" * 60)

RU  : Привет, как дела?
REF : Hi, how are you?
PRED: Hi, how are you doing?
------------------------------------------------------------
RU  : Я люблю читать книги.
REF : I love reading books.
PRED: I like reading books.
------------------------------------------------------------
RU  : Сегодня хорошая погода.
REF : The weather is nice today.
PRED: Today the weather is good weather today.
------------------------------------------------------------
RU  : Он не пришёл на встречу.
REF : He didn't come to the meeting.
PRED: He didn't come to the meeting.
------------------------------------------------------------
RU  : Завтра мы поедем в Москву.
REF : We will go to Moscow tomorrow.
PRED: Tomorrow we'll go to Moscow for Moscow.
------------------------------------------------------------
RU  : Мы поедем в Москву завтра.
REF : We will go to Moscow tomorrow.
PRED: We'll go to Moscow tomorrow morning.
------------------------------------------------------------
RU  : Она готовит ужин на кухн